# Machine Sensor Forecasting

## Project Overview

Forecasts **daily machine operating temperature** (°C) over a 14-day horizon. Synthetic data spans 2 years with weekly utilization cycles, seasonal ambient effects, and gradual degradation.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily machine temperature readings, predict the next 14 days. Predictive maintenance systems need sensor forecasts to detect anomalies and schedule maintenance before failures occur.

## Why This Project Matters

Unplanned machine failures cost manufacturing plants $50B+ annually. Forecasting sensor values enables anomaly detection — when actual readings deviate significantly from forecasts, it signals potential failure. Early detection saves costly emergency repairs and production downtime.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily average machine temperature
- Weekly utilization cycle (higher on production days)
- Seasonal ambient temperature effect
- Gradual upward trend (bearing degradation / wear)
- Occasional thermal spikes (overload events)

| Column | Description |
|--------|-------------|
| `date` | Date |
| `temperature_c` | Daily average machine temperature (°C) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'temperature_c'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

base = 65 + 0.005 * t  # gradual degradation (bearing wear)
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow <= 4: weekly[i] = 3  # production heat
    elif dow == 5: weekly[i] = -2  # reduced load
    else: weekly[i] = -5  # idle/cooling

# Ambient temperature effect
ambient = 4 * np.sin(2 * np.pi * (t - 80) / 365)

# Thermal spikes (overload events)
spikes = np.zeros(N_POINTS)
for s in range(0, N_POINTS, 150):
    spike_day = min(s + np.random.randint(30, 90), N_POINTS - 1)
    spikes[spike_day] = 8 + np.random.uniform(0, 5)

noise = np.random.normal(0, 1.5, N_POINTS)
temperature_c = base + weekly + ambient + spikes + noise
temperature_c = np.maximum(temperature_c, 40).round(1)

df = pd.DataFrame({'date': dates, 'temperature_c': temperature_c})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,temperature_c
0,2022-01-01,61.4
1,2022-01-02,57.2
2,2022-01-03,63.4
3,2022-01-04,64.9
4,2022-01-05,63.5


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('temperature_c Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("temperature_c Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

temperature_c Statistics:
count    730.000000
mean      67.998767
std        4.555647
min       53.700000
25%       65.000000
50%       68.100000
75%       71.400000
max       84.900000
Name: temperature_c, dtype: float64

CV: 0.067
Skewness: -0.263


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:        4.0 | RMSE:        4.3 | MAPE:  6.04%
Seasonal Naive (7)             | MAE:        1.2 | RMSE:        1.5 | MAPE:  1.81%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                           Adjusted R-Squared   R-Squared       RMSE  Time Taken
Model                                                                           
KernelRidge                       4679.851202 -358.911631  68.838985    0.051211
GaussianProcessRegressor           219.091631  -15.776279  14.862242    0.044696
MLPRegressor                        38.792650   -1.907127   6.186835    0.285608
QuantileRegressor                   21.939538   -0.610734   4.605199    0.057147
DummyRegressor                      21.066045   -0.543542   4.508123    0.008405
DecisionTreeRegressor               12.525174    0.113448   3.416556    0.015208
ExtraTreeRegressor                  10.334671    0.281948   3.074782    0.011800
OrthogonalMatchingPursuit            7.725704    0.482638   2.609959    0.007906
Lasso                                7.388986    0.508540   2.543787    0.011886
LassoLars                            7.388881    0.508548   2.543767    0.009024


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: catboost
FLAML (catboost)               | MAE:        1.4 | RMSE:        1.8 | MAPE:  2.09%
FLAML Test (catboost)          | MAE:        1.1 | RMSE:        1.3 | MAPE:  1.71%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:        1.1 | RMSE:        1.3 | MAPE:  1.74%
SF AutoETS                     | MAE:        1.1 | RMSE:        1.3 | MAPE:  1.71%
SF SeasonalNaive               | MAE:        1.4 | RMSE:        1.9 | MAPE:  2.19%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                model      MAE     RMSE     MAPE
           SF AutoETS 1.120247 1.284727 1.710898
FLAML Test (catboost) 1.122170 1.303626 1.712963
         SF AutoARIMA 1.137531 1.310790 1.741098
   Seasonal Naive (7) 1.171429 1.532505 1.812783
     FLAML (catboost) 1.352079 1.806330 2.089273
     SF SeasonalNaive 1.407143 1.865284 2.187622
   Naive (Last Value) 4.007143 4.337132 6.042415

Top 3:
                model      MAE     RMSE     MAPE
           SF AutoETS 1.120247 1.284727 1.710898
FLAML Test (catboost) 1.122170 1.303626 1.712963
         SF AutoARIMA 1.137531 1.310790 1.741098


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 0.28, Std: 1.27


## Interpretation and Business Insight

- Machine temperature reflects utilization patterns (higher on production days)
- Seasonal ambient temperature adds a predictable cycle
- Gradual upward trend signals bearing wear / degradation
- Thermal spikes indicate overload events
- Forecast deviations from actual readings signal potential failures

## Limitations

1. Synthetic — real sensor data has much higher frequency (seconds/minutes)
2. Only one sensor — real machines have vibration, pressure, current, etc.
3. Daily averages mask intraday peaks
4. No maintenance event labels
5. No load/production rate correlation

## How to Improve This Project

1. Use high-frequency sensor data (per-second or per-minute)
2. Add multiple sensor channels (vibration, pressure, current)
3. Include production load as a feature
4. Label maintenance events for supervised anomaly detection
5. Use anomalib or PyOD for anomaly scoring

## Production Considerations

- Real-time sensor monitoring with forecast-based anomaly alerts
- Integration with CMMS (computerized maintenance management)
- Remaining useful life (RUL) estimation
- Maintenance work order auto-generation

## Common Mistakes

1. Using daily averages when sub-second resolution is available
2. Not accounting for load changes when interpreting temperature
3. Setting fixed thresholds instead of forecast-based anomaly detection
4. Ignoring gradual degradation trends

## Mini Challenge / Exercises

1. Detect the degradation trend and estimate when temperature exceeds a threshold
2. Add synthetic vibration data and build a multi-sensor model
3. Create a simple anomaly detector: flag days where actual > forecast + 2σ
4. Build a remaining useful life (RUL) estimate from the degradation trend

## Final Summary / Key Takeaways

1. Machine sensor forecasting enables predictive maintenance
2. Temperature reflects both utilization and machine health
3. Gradual degradation trends signal future failures
4. Forecast-based anomaly detection is more effective than fixed thresholds
5. Real deployment needs high-frequency, multi-sensor models

In [18]:
import json
metrics = {
    'project': 'Machine Sensor Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Machine Sensor Forecasting — Complete ===")

Metrics saved.

=== Machine Sensor Forecasting — Complete ===
